# Chess Policy Model — Data Preparation (V2)

This notebook builds the ChessMind Arena policy training dataset using PAWN Stockfish 18 annotations.

Source:
`thomas-schweich/pawn-stockfish-100m`

Configuration:
`nodes_1024`

Each training example represents one chess position and contains:

- FEN
- Search-ranked candidate moves in UCI notation
- Candidate centipawn evaluations
- Candidate ranks
- Target move (highest-ranked candidate)
- NNUE evaluations for all legal moves
- Game and engine metadata

Target: 100,000 positions

Splits:
- 80% training
- 10% validation
- 10% test

Positions are reconstructed by replaying complete games.
The original PAWN action vocabulary is decoded into UCI notation.

In [1]:
!pip install -q python-chess pandas pyarrow tqdm datasets huggingface_hub requests

In [2]:
import os
import json
import requests

from pathlib import Path
from tqdm.auto import tqdm

import chess

from datasets import load_dataset, Dataset, DatasetDict
from huggingface_hub import login
import getpass

HF_TOKEN = getpass.getpass("Enter your Hugging Face token: ")
login(token=HF_TOKEN)

os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [3]:
SEED = 42

SOURCE_DATASET = "thomas-schweich/pawn-stockfish-100m"
SOURCE_CONFIG = "nodes_1024"

TARGET_POSITIONS = 100_000  # Pilot; increase to 100,000 after validation

SPLIT_RATIOS = {
    "train": 0.80,
    "validation": 0.10,
    "test": 0.10,
}

HF_USERNAME = "Mridul-Dixit"
OUTPUT_DATASET = "chessmind-arena-policy-v2"

DATA_DIR = Path("ml/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

VOCAB_FILE = DATA_DIR / "searchless_chess_vocabulary.json"

SPLIT_TARGETS = {
    split: int(TARGET_POSITIONS * ratio)
    for split, ratio in SPLIT_RATIOS.items()
}

print("Target positions:", TARGET_POSITIONS)
print("Split targets:", SPLIT_TARGETS)

Target positions: 100000
Split targets: {'train': 80000, 'validation': 10000, 'test': 10000}


In [4]:
VOCAB_URL = (
    "https://raw.githubusercontent.com/"
    "thomas-schweich/PAWN/main/"
    "searchless_chess_vocabulary.json"
)

if not VOCAB_FILE.exists():
    response = requests.get(VOCAB_URL, timeout=60)
    response.raise_for_status()

    VOCAB_FILE.write_text(response.text)

with open(VOCAB_FILE, "r") as file:
    vocabulary = json.load(file)

move_to_action = vocabulary["move_to_action"]

action_to_move = {
    int(action_id): move
    for move, action_id in move_to_action.items()
}

assert len(action_to_move) == 1968
assert set(action_to_move.keys()) == set(range(1968))

print("Loaded actions:", len(action_to_move))
print("Example mappings:")

for action_id in [317, 346, 1442]:
    print(action_id, "->", action_to_move[action_id])

Loaded actions: 1968
Example mappings:
317 -> e2e4
346 -> f2f4
1442 -> c7c6


In [5]:
source_dataset = load_dataset(
    SOURCE_DATASET,
    SOURCE_CONFIG,
    streaming=True,
)

print("Available source splits:", list(source_dataset.keys()))

required_splits = {"train", "validation", "test"}

if not required_splits.issubset(set(source_dataset.keys())):
    raise ValueError(
        f"Expected source splits {required_splits}, "
        f"but found {list(source_dataset.keys())}"
    )

Resolving data files:   0%|          | 0/553 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/321 [00:00<?, ?it/s]

Available source splits: ['train', 'validation', 'test']


In [6]:
def decode_evaluations(evaluations, board):
    decoded = []

    for rank, item in enumerate(evaluations or [], start=1):
        action_id = int(item["move_idx"])

        if action_id not in action_to_move:
            raise ValueError(f"Unknown PAWN action ID: {action_id}")

        move_uci = action_to_move[action_id]
        move = chess.Move.from_uci(move_uci)

        if move not in board.legal_moves:
            raise ValueError(
                f"Illegal move {move_uci} for position {board.fen()}"
            )

        decoded.append({
            "rank": rank,
            "move": move_uci,
            "score_cp": item["score_cp"],
        })

    return decoded

In [7]:
def decode_nnue_evaluations(evaluations, board):
    decoded = []

    for item in evaluations or []:
        action_id = int(item["move_idx"])

        if action_id not in action_to_move:
            raise ValueError(f"Unknown PAWN action ID: {action_id}")

        move_uci = action_to_move[action_id]
        move = chess.Move.from_uci(move_uci)

        if move not in board.legal_moves:
            raise ValueError(
                f"Illegal NNUE move {move_uci} for position {board.fen()}"
            )

        decoded.append({
            "move": move_uci,
            "score_cp": item["score_cp"],
            "score_eval_v": item["score_eval_v"],
            "score_psqt": item["score_psqt"],
            "score_positional": item["score_positional"],
        })

    return decoded

In [8]:
def extract_game_positions(game, source_split):
    board = chess.Board()

    uci_moves = game["uci"]
    tokens = game["tokens"]
    cp_evals = game["cp_evals"]
    nnue_evals = game["nnue_evals"]

    game_length = int(game["game_length"])

    if not (
        len(uci_moves)
        == len(tokens)
        == len(cp_evals)
        == len(nnue_evals)
        == game_length
    ):
        raise ValueError("Inconsistent game sequence lengths")

    game_id = str(game["global_game_index"])

    positions = []

    for ply in range(game_length):
        actual_uci = uci_moves[ply]
        actual_move = chess.Move.from_uci(actual_uci)

        if actual_move not in board.legal_moves:
            raise ValueError(
                f"Illegal played move {actual_uci} in game {game_id}"
            )

        # Verify PAWN token mapping against its UCI sequence.
        token_uci = action_to_move[int(tokens[ply])]

        if token_uci != actual_uci:
            raise ValueError(
                f"Token mismatch: {token_uci} != {actual_uci}"
            )

        fen = board.fen()

        # Search-derived candidate evaluations.
        candidates = decode_evaluations(
            cp_evals[ply],
            board,
        )

        # Dense NNUE evaluations for all legal moves.
        nnue_candidates = decode_nnue_evaluations(
            nnue_evals[ply],
            board,
        )

        if candidates:
            positions.append({
                "fen": fen,
                "target_move": candidates[0]["move"],
                "candidates": candidates,
                "nnue_evaluations": nnue_candidates,

                "game_id": game_id,
                "ply": ply,
                "result": game["result"],
                "source_split": source_split,

                "nodes": game["nodes"],
                "multi_pv": game["multi_pv"],
                "opening_multi_pv": game["opening_multi_pv"],
                "opening_plies": game["opening_plies"],
                "stockfish_version": game["stockfish_version"],
            })

        # Advance using the actual played game move.
        board.push(actual_move)

    return positions

In [9]:
import time

split_records = {}
seen_fens = set()

for split_name, target_count in SPLIT_TARGETS.items():

    print(f"\nGenerating {split_name} split...")

    records = []
    processed_games = 0
    skipped_games = 0

    stream = source_dataset[split_name].shuffle(
        seed=SEED,
        buffer_size=10_000,
    )

    start_time = time.time()

    with tqdm(
        total=target_count,
        desc=f"{split_name} positions",
        unit="positions",
    ) as pbar:

        for game in stream:

            if len(records) >= target_count:
                break

            processed_games += 1

            try:
                game_positions = extract_game_positions(
                    game,
                    split_name,
                )

                for position in game_positions:

                    if len(records) >= target_count:
                        break

                    fen = position["fen"]

                    if fen in seen_fens:
                        continue

                    records.append(position)
                    seen_fens.add(fen)

                    pbar.update(1)

                pbar.set_postfix(
                    games=processed_games,
                    skipped=skipped_games,
                )

            except (ValueError, TypeError, KeyError) as error:
                skipped_games += 1
                continue

    elapsed = time.time() - start_time

    split_records[split_name] = records

    print(f"\n{split_name} summary:")
    print("Positions collected:", len(records))
    print("Games processed:", processed_games)
    print("Games skipped:", skipped_games)
    print(f"Time taken: {elapsed / 60:.2f} minutes")

    if len(records) < target_count:
        raise RuntimeError(
            f"Could not collect enough positions for {split_name}"
        )


Generating train split...


train positions:   0%|          | 0/80000 [00:00<?, ?positions/s]


train summary:
Positions collected: 80000
Games processed: 635
Games skipped: 0
Time taken: 1.95 minutes

Generating validation split...


validation positions:   0%|          | 0/10000 [00:00<?, ?positions/s]


validation summary:
Positions collected: 10000
Games processed: 89
Games skipped: 0
Time taken: 0.50 minutes

Generating test split...


test positions:   0%|          | 0/10000 [00:00<?, ?positions/s]


test summary:
Positions collected: 10000
Games processed: 78
Games skipped: 0
Time taken: 0.48 minutes


In [10]:
dataset_dict = DatasetDict({
    split: Dataset.from_list(records)
    for split, records in split_records.items()
})

print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['fen', 'target_move', 'candidates', 'nnue_evaluations', 'game_id', 'ply', 'result', 'source_split', 'nodes', 'multi_pv', 'opening_multi_pv', 'opening_plies', 'stockfish_version'],
        num_rows: 80000
    })
    validation: Dataset({
        features: ['fen', 'target_move', 'candidates', 'nnue_evaluations', 'game_id', 'ply', 'result', 'source_split', 'nodes', 'multi_pv', 'opening_multi_pv', 'opening_plies', 'stockfish_version'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['fen', 'target_move', 'candidates', 'nnue_evaluations', 'game_id', 'ply', 'result', 'source_split', 'nodes', 'multi_pv', 'opening_multi_pv', 'opening_plies', 'stockfish_version'],
        num_rows: 10000
    })
})


In [11]:
def validate_record(record):
    board = chess.Board(record["fen"])

    if not board.is_valid():
        return False

    target = chess.Move.from_uci(record["target_move"])

    if target not in board.legal_moves:
        return False

    if not record["candidates"]:
        return False

    if record["candidates"][0]["move"] != record["target_move"]:
        return False

    for candidate in record["candidates"]:
        move = chess.Move.from_uci(candidate["move"])

        if move not in board.legal_moves:
            return False

    for candidate in record["nnue_evaluations"]:
        move = chess.Move.from_uci(candidate["move"])

        if move not in board.legal_moves:
            return False

    return True


for split_name, split_dataset in dataset_dict.items():

    valid = [
        validate_record(record)
        for record in split_dataset
    ]

    print(
        split_name,
        "valid:",
        sum(valid),
        "/",
        len(valid),
    )

    assert all(valid)

train valid: 80000 / 80000
validation valid: 10000 / 10000
test valid: 10000 / 10000


In [12]:
for split_name, split_dataset in dataset_dict.items():

    candidate_counts = [
        len(record["candidates"])
        for record in split_dataset
    ]

    nnue_counts = [
        len(record["nnue_evaluations"])
        for record in split_dataset
    ]

    print(f"\n{split_name.upper()}")
    print("Positions:", len(split_dataset))
    print("Average search candidates:", sum(candidate_counts) / len(candidate_counts))
    print("Average NNUE moves:", sum(nnue_counts) / len(nnue_counts))
    print("Unique FENs:", len(set(split_dataset["fen"])))


TRAIN
Positions: 80000
Average search candidates: 4.801225
Average NNUE moves: 27.1648125
Unique FENs: 80000

VALIDATION
Positions: 10000
Average search candidates: 4.8082
Average NNUE moves: 27.8789
Unique FENs: 10000

TEST
Positions: 10000
Average search candidates: 4.7971
Average NNUE moves: 26.6387
Unique FENs: 10000


In [13]:
sample = dataset_dict["train"][0]

print("FEN:")
print(sample["fen"])

print("\nTarget move:")
print(sample["target_move"])

print("\nSearch candidates:")
for candidate in sample["candidates"]:
    print(candidate)

print("\nFirst five NNUE evaluations:")
for candidate in sample["nnue_evaluations"][:5]:
    print(candidate)

FEN:
rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1

Target move:
e2e4

Search candidates:
{'move': 'e2e4', 'rank': 1, 'score_cp': 18}
{'move': 'c2c4', 'rank': 2, 'score_cp': 1}
{'move': 'd2d3', 'rank': 3, 'score_cp': -2}
{'move': 'a2a3', 'rank': 4, 'score_cp': -3}
{'move': 'a2a4', 'rank': 5, 'score_cp': -7}
{'move': 'b2b3', 'rank': 6, 'score_cp': -8}
{'move': 'h2h3', 'rank': 7, 'score_cp': -12}
{'move': 'b1c3', 'rank': 8, 'score_cp': -14}
{'move': 'b1a3', 'rank': 9, 'score_cp': -17}
{'move': 'f2f4', 'rank': 10, 'score_cp': -42}
{'move': 'g1h3', 'rank': 11, 'score_cp': -54}
{'move': 'b2b4', 'rank': 12, 'score_cp': -59}
{'move': 'h2h4', 'rank': 13, 'score_cp': -64}
{'move': 'f2f3', 'rank': 14, 'score_cp': -85}
{'move': 'g2g4', 'rank': 15, 'score_cp': -100}
{'move': 'd2d4', 'rank': 16, 'score_cp': 17}
{'move': 'e2e3', 'rank': 17, 'score_cp': 11}
{'move': 'c2c3', 'rank': 18, 'score_cp': 10}
{'move': 'g2g3', 'rank': 19, 'score_cp': 4}
{'move': 'g1f3', 'rank': 20, 'score_cp': 1}



In [14]:
repo_id = f"{HF_USERNAME}/{OUTPUT_DATASET}"

dataset_dict.push_to_hub(repo_id)

print("Uploaded dataset:", repo_id)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpdqfhe5lh.parquet    :   3%|2         | 2.98MB / 99.4MB            

... (12 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpqpov_qxx.parquet    :  28%|##7       | 3.39MB / 12.2MB            

Validating                    :           |   0%            

Uploaded dataset: Mridul-Dixit/chessmind-arena-policy-v2
